In [26]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os
import pickle
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from datetime import datetime

In [27]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense, Concatenate,
    Dropout, BatchNormalization
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers.legacy import Adam

In [28]:
# Verify tensorflow-metal is active

print(f"TensorFlow Version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.list_physical_devices('GPU'))}")
print(f"Physical Devices: {tf.config.list_physical_devices()}")

TensorFlow Version: 2.15.0
Num GPUs Available: 1
Physical Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [29]:
DATA_DIR   = '/Users/ruben/Desktop/Thesis/TrainingData/final-data/output'
OUTPUT_DIR = DATA_DIR
SEED       = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

In [30]:
# Load merged arrays
print("\nLoading merged dataset...")
X_dynamic = np.load(f'{DATA_DIR}/X_dynamic.npy')
X_static  = np.load(f'{DATA_DIR}/X_static.npy')
y         = np.load(f'{DATA_DIR}/y_wealth.npy')

print(f"X_dynamic : {X_dynamic.shape}")
print(f"X_static  : {X_static.shape}")
print(f"y         : {y.shape}")
print(f"y range   : [{y.min():.4f}, {y.max():.4f}]")



Loading merged dataset...
X_dynamic : (1234, 4, 4096)
X_static  : (1234, 20)
y         : (1234,)
y range   : [-2.3689, 1.8740]


In [31]:
from sklearn.decomposition import PCA

N_COMPONENTS = 256  # experiment with 64, 128, 256

# Reshape from (1234, 4, 4096) to (1234*4, 4096) so PCA sees each
# timestep as an independent sample
X_dyn_flat = X_dynamic.reshape(-1, 4096)

# Fit PCA on all data first (we will refit per fold during CV)
pca_global = PCA(n_components=N_COMPONENTS, random_state=SEED)
pca_global.fit(X_dyn_flat)

print(f"Explained variance retained: "
      f"{pca_global.explained_variance_ratio_.sum():.4f}")

# Transform and reshape back to (1234, 4, N_COMPONENTS)
X_dynamic_pca = pca_global.transform(X_dyn_flat).reshape(
    X_dynamic.shape[0], 4, N_COMPONENTS)

print(f"X_dynamic reduced: {X_dynamic_pca.shape}")

Explained variance retained: 0.9087
X_dynamic reduced: (1234, 4, 256)


In [32]:
from sklearn.model_selection import train_test_split

# Hold out 20% for hyperparameter tuning
X_dyn_train_tune, X_dyn_val_tune, X_stat_train_tune, X_stat_val_tune, y_train_tune, y_val_tune = train_test_split(
    X_dynamic, X_static, y, test_size=0.2, random_state=SEED
)

print(f"Tuning train size: {len(y_train_tune)}")
print(f"Tuning val size:   {len(y_val_tune)}")

Tuning train size: 987
Tuning val size:   247


In [33]:
from tensorflow.keras.optimizers.legacy import Adam
from tensorflow.keras.callbacks import EarlyStopping

def build_model_with_params(lstm_units, dropout_dyn, dense_static_units, dropout_stat,
                            fusion_units, dropout_fusion, l2_reg, lr):
    # Dynamic branch
    input_dynamic = Input(shape=(4, N_COMPONENTS), name='Dynamic_Input')
    x1 = LSTM(lstm_units, return_sequences=False, kernel_regularizer=l2(l2_reg))(input_dynamic)
    x1 = Dropout(dropout_dyn)(x1)
    
    # Static branch
    input_static = Input(shape=(X_static.shape[1],), name='Static_Input')
    x2 = Dense(dense_static_units, activation='relu', kernel_regularizer=l2(l2_reg))(input_static)
    x2 = BatchNormalization()(x2)
    x2 = Dropout(dropout_stat)(x2)
    
    # Fusion
    combined = Concatenate()([x1, x2])
    z = Dense(fusion_units, activation='relu', kernel_regularizer=l2(l2_reg))(combined)
    z = Dropout(dropout_fusion)(z)
    out = Dense(1, activation='linear')(z)
    
    model = Model(inputs=[input_dynamic, input_static], outputs=out)
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model

In [34]:
import itertools

# Define parameter grid
param_grid = {
    'lstm_units': [32, 64, 96],
    'dropout_dyn': [0.3, 0.4, 0.5],
    'dense_static_units': [16, 32, 64],
    'dropout_stat': [0.2, 0.3, 0.4],
    'fusion_units': [16, 32, 64],
    'dropout_fusion': [0.2, 0.3, 0.4],
    'l2_reg': [1e-5, 1e-4, 1e-3],
    'lr': [1e-3, 5e-4, 1e-4]
}

# Generate all combinations (may be many – reduce if needed)
keys, values = zip(*param_grid.items())
all_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"Total combinations to try: {len(all_combinations)}")

# Optionally limit to a random subset for speed
import random
random.seed(SEED)
n_trials = 30   # adjust based on your patience
combinations = random.sample(all_combinations, n_trials)

pca_tune = PCA(n_components=N_COMPONENTS, random_state=SEED)
pca_tune.fit(X_dyn_train_tune.reshape(-1, 4096))
X_dyn_train_tune_pca = pca_tune.transform(
    X_dyn_train_tune.reshape(-1, 4096)).reshape(-1, 4, N_COMPONENTS)
X_dyn_val_tune_pca = pca_tune.transform(
    X_dyn_val_tune.reshape(-1, 4096)).reshape(-1, 4, N_COMPONENTS)

best_val_loss = float('inf')
best_params = None
results = []

for i, params in enumerate(combinations):
    print(f"\nTrial {i+1}/{n_trials}: {params}")
    
    # Scale static features on the tuning train split
    scaler_tune = StandardScaler()
    X_stat_train_scaled = scaler_tune.fit_transform(X_stat_train_tune)
    X_stat_val_scaled = scaler_tune.transform(X_stat_val_tune)
    
    # Build model with current params
    model = build_model_with_params(**params)
    
    # Early stopping
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    
    history = model.fit(
        x=[X_dyn_train_tune_pca, X_stat_train_scaled],
        y=y_train_tune,
        validation_data=([X_dyn_val_tune_pca, X_stat_val_scaled], y_val_tune),
        epochs=80,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )
    
    # Best validation loss from early stopping
    val_loss = min(history.history['val_loss'])
    results.append({'params': params, 'val_loss': val_loss})
    
    print(f"  -> Best val loss: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_params = params
        print(f"  *** New best! ***")

print("\n" + "="*50)
print("BEST HYPERPARAMETERS:")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"Best validation loss: {best_val_loss:.4f}")

Total combinations to try: 6561

Trial 1/30: {'lstm_units': 96, 'dropout_dyn': 0.4, 'dense_static_units': 16, 'dropout_stat': 0.3, 'fusion_units': 64, 'dropout_fusion': 0.2, 'l2_reg': 1e-05, 'lr': 0.001}
  -> Best val loss: 0.2514
  *** New best! ***

Trial 2/30: {'lstm_units': 32, 'dropout_dyn': 0.4, 'dense_static_units': 16, 'dropout_stat': 0.4, 'fusion_units': 16, 'dropout_fusion': 0.4, 'l2_reg': 0.0001, 'lr': 0.001}
  -> Best val loss: 0.2905

Trial 3/30: {'lstm_units': 32, 'dropout_dyn': 0.3, 'dense_static_units': 16, 'dropout_stat': 0.4, 'fusion_units': 32, 'dropout_fusion': 0.3, 'l2_reg': 0.001, 'lr': 0.001}
  -> Best val loss: 0.3125

Trial 4/30: {'lstm_units': 96, 'dropout_dyn': 0.5, 'dense_static_units': 16, 'dropout_stat': 0.4, 'fusion_units': 64, 'dropout_fusion': 0.4, 'l2_reg': 0.001, 'lr': 0.0001}
  -> Best val loss: 0.6078

Trial 5/30: {'lstm_units': 64, 'dropout_dyn': 0.3, 'dense_static_units': 16, 'dropout_stat': 0.2, 'fusion_units': 64, 'dropout_fusion': 0.3, 'l2_reg'

In [35]:
# Best params found from grid search (update these values)
BEST_PARAMS = best_params

def build_model(n_static, n_dynamic_features=N_COMPONENTS):
    # Dynamic branch
    input_dynamic = Input(shape=(4, n_dynamic_features), name='Dynamic_Input')
    x1 = LSTM(BEST_PARAMS['lstm_units'], return_sequences=False,
              kernel_regularizer=l2(BEST_PARAMS['l2_reg']))(input_dynamic)
    x1 = Dropout(BEST_PARAMS['dropout_dyn'])(x1)
    
    # Static branch
    input_static = Input(shape=(n_static,), name='Static_Input')
    x2 = Dense(BEST_PARAMS['dense_static_units'], activation='relu',
               kernel_regularizer=l2(BEST_PARAMS['l2_reg']))(input_static)
    x2 = BatchNormalization()(x2)
    x2 = Dropout(BEST_PARAMS['dropout_stat'])(x2)
    
    # Fusion
    combined = Concatenate()([x1, x2])
    z = Dense(BEST_PARAMS['fusion_units'], activation='relu',
              kernel_regularizer=l2(BEST_PARAMS['l2_reg']))(combined)
    z = Dropout(BEST_PARAMS['dropout_fusion'])(z)
    out = Dense(1, activation='linear', name='Wealth_Prediction')(z)
    
    model = Model(inputs=[input_dynamic, input_static], outputs=out)
    model.compile(optimizer=Adam(learning_rate=BEST_PARAMS['lr']),
                  loss='mse', metrics=['mae'])
    return model

In [36]:
# ── K-FOLD CROSS VALIDATION ────────────────────────────
N_FOLDS = 5
kf      = KFold(
    n_splits   = N_FOLDS,
    shuffle    = True,
    random_state = SEED
)

fold_metrics  = []
all_y_true    = []
all_y_pred    = []

print(f"\n{'='*50}")
print(f"K-FOLD CROSS VALIDATION ({N_FOLDS} folds)")
print(f"{'='*50}")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_dynamic)):

    print(f"\nFold {fold+1}/{N_FOLDS} | "
          f"Train: {len(train_idx)} | Val: {len(val_idx)}")

    # Split raw (pre-PCA) dynamic data
    X_dyn_tr_raw  = X_dynamic[train_idx]   # (n_train, 4, 4096)
    X_dyn_val_raw = X_dynamic[val_idx]

    # Fit PCA on training fold only
    pca = PCA(n_components=N_COMPONENTS, random_state=SEED)
    pca.fit(X_dyn_tr_raw.reshape(-1, 4096))

    # Transform both splits
    X_dyn_tr = pca.transform(
        X_dyn_tr_raw.reshape(-1, 4096)).reshape(-1, 4, N_COMPONENTS)
    X_dyn_val = pca.transform(
        X_dyn_val_raw.reshape(-1, 4096)).reshape(-1, 4, N_COMPONENTS)

    # Scale static features (same as before)
    X_stat_tr  = X_static[train_idx]
    X_stat_val = X_static[val_idx]
    y_tr       = y[train_idx]
    y_val      = y[val_idx]

    scaler     = StandardScaler()
    X_stat_tr  = scaler.fit_transform(X_stat_tr)
    X_stat_val = scaler.transform(X_stat_val)

    # Build and train
    model = build_model(X_static.shape[1], N_COMPONENTS)

    callbacks = [
        EarlyStopping(
            monitor              = 'val_loss',
            patience             = 15,
            restore_best_weights = True,
            verbose              = 0
        ),
        ReduceLROnPlateau(
            monitor  = 'val_loss',
            factor   = 0.5,
            patience = 7,
            min_lr   = 1e-6,
            verbose  = 0
        )
    ]

    model.fit(
        x               = [X_dyn_tr, X_stat_tr],
        y               = y_tr,
        validation_data = ([X_dyn_val, X_stat_val], y_val),
        epochs          = 200,
        batch_size      = 32,
        callbacks       = callbacks,
        verbose         = 0
    )

    # Evaluate
    preds   = model.predict(
        [X_dyn_val, X_stat_val], verbose=0).flatten()
    r2      = r2_score(y_val, preds)
    rmse    = np.sqrt(np.mean((y_val - preds) ** 2))
    pearson = pearsonr(y_val, preds)[0]

    fold_metrics.append({
        'fold': fold + 1,
        'R2'  : round(r2, 4),
        'RMSE': round(rmse, 4),
        'r'   : round(pearson, 4)
    })
    all_y_true.extend(y_val.tolist())
    all_y_pred.extend(preds.tolist())

    print(f"  R²={r2:.4f}  RMSE={rmse:.4f}  r={pearson:.4f}")
    
    tf.keras.backend.clear_session()

# CV Summary
df_cv = pd.DataFrame(fold_metrics)
print(f"\n{'='*50}")
print("CROSS VALIDATION RESULTS")
print(f"{'='*50}")
print(df_cv.to_string(index=False))
print(f"\nMean R²  : {df_cv['R2'].mean():.4f} "
      f"± {df_cv['R2'].std():.4f}")
print(f"Mean RMSE: {df_cv['RMSE'].mean():.4f} "
      f"± {df_cv['RMSE'].std():.4f}")
print(f"Mean r   : {df_cv['r'].mean():.4f} "
      f"± {df_cv['r'].std():.4f}")

# Pooled metrics
y_true_arr = np.array(all_y_true)
y_pred_arr = np.array(all_y_pred)
print(f"\nPooled R²  : "
      f"{r2_score(y_true_arr, y_pred_arr):.4f}")
print(f"Pooled RMSE: "
      f"{np.sqrt(np.mean((y_true_arr-y_pred_arr)**2)):.4f}")
print(f"Pooled r   : "
      f"{pearsonr(y_true_arr, y_pred_arr)[0]:.4f}")

df_cv.to_csv(
    f'{OUTPUT_DIR}/kfold_cv_results.csv', index=False)


K-FOLD CROSS VALIDATION (5 folds)

Fold 1/5 | Train: 987 | Val: 247
  R²=0.4219  RMSE=0.5172  r=0.6666

Fold 2/5 | Train: 987 | Val: 247
  R²=0.4418  RMSE=0.4854  r=0.6695

Fold 3/5 | Train: 987 | Val: 247
  R²=0.4703  RMSE=0.5197  r=0.6897

Fold 4/5 | Train: 987 | Val: 247
  R²=0.4413  RMSE=0.5218  r=0.6664

Fold 5/5 | Train: 988 | Val: 246
  R²=0.4493  RMSE=0.5106  r=0.6791

CROSS VALIDATION RESULTS
 fold     R2   RMSE      r
    1 0.4219 0.5172 0.6666
    2 0.4418 0.4854 0.6695
    3 0.4703 0.5197 0.6897
    4 0.4413 0.5218 0.6664
    5 0.4493 0.5106 0.6791

Mean R²  : 0.4449 ± 0.0174
Mean RMSE: 0.5109 ± 0.0149
Mean r   : 0.6743 ± 0.0101

Pooled R²  : 0.4479
Pooled RMSE: 0.5111
Pooled r   : 0.6693


In [37]:
# ── FINAL MODEL ON ALL DATA ────────────────────────────
# ── RETRAIN FINAL MODEL CORRECTLY ─────────────────────
import os

print(f"\n{'='*50}")
print("RETRAINING FINAL MODEL (corrected)")
print(f"{'='*50}")
print(f"Best params: {BEST_PARAMS}")

# Scale static features
scaler_final    = StandardScaler()
X_static_scaled = scaler_final.fit_transform(X_static)

# Apply PCA (fit on all data)
pca_final  = PCA(n_components=N_COMPONENTS, random_state=SEED)
X_dyn_all  = pca_final.fit_transform(
    X_dynamic.reshape(-1, 4096)
).reshape(-1, 4, N_COMPONENTS)

print(f"PCA variance retained: "
      f"{pca_final.explained_variance_ratio_.sum():.4f}")

final_model = build_model(X_static.shape[1])

callbacks_final = [
    EarlyStopping(
        monitor              = 'val_loss',
        patience             = 20,
        restore_best_weights = True,  # saves best, not last
        verbose              = 1
    ),
    ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 8,
        min_lr   = 1e-6,
        verbose  = 1
    ),
    ModelCheckpoint(
        filepath       = f'{OUTPUT_DIR}/final_hybrid_poverty_model_best.keras',
        monitor        = 'val_loss',
        save_best_only = True,
        verbose        = 0
    )
]

history_final = final_model.fit(
    x                = [X_dyn_all, X_static_scaled],
    y                = y,
    validation_split = 0.1,    # 10% held out to monitor overfitting
    epochs           = 300,    # high ceiling, EarlyStopping will stop it
    batch_size       = 32,
    callbacks        = callbacks_final,
    verbose          = 1
)

best_epoch    = np.argmin(history_final.history['val_loss']) + 1
best_val_loss = min(history_final.history['val_loss'])
print(f"\nBest epoch    : {best_epoch}")
print(f"Best val_loss : {best_val_loss:.4f}")

# Save final model, scaler, and PCA
final_model.save(
    f'{OUTPUT_DIR}/final_hybrid_poverty_model.keras')
with open(f'{OUTPUT_DIR}/final_scaler.pkl', 'wb') as f:
    pickle.dump(scaler_final, f)
with open(f'{OUTPUT_DIR}/final_pca.pkl', 'wb') as f:
    pickle.dump(pca_final, f)

print(f"\nSaved:")
print(f"  final_hybrid_poverty_model.keras")
print(f"  final_scaler.pkl")
print(f"  final_pca.pkl  ← NEW: needed for inference")
print(f"\nNext: SHAP computation")


RETRAINING FINAL MODEL (corrected)
Best params: {'lstm_units': 96, 'dropout_dyn': 0.5, 'dense_static_units': 16, 'dropout_stat': 0.4, 'fusion_units': 64, 'dropout_fusion': 0.4, 'l2_reg': 1e-05, 'lr': 0.0005}
PCA variance retained: 0.9087
Epoch 1/300
35/35 [==============================] - 3s 47ms/step - loss: 1.3765 - mae: 0.8961 - val_loss: 0.5411 - val_mae: 0.5757 - lr: 5.0000e-04
Epoch 2/300
35/35 [==============================] - 1s 21ms/step - loss: 0.9881 - mae: 0.7730 - val_loss: 0.4717 - val_mae: 0.5202 - lr: 5.0000e-04
Epoch 3/300
35/35 [==============================] - 1s 20ms/step - loss: 0.7480 - mae: 0.6761 - val_loss: 0.4559 - val_mae: 0.5058 - lr: 5.0000e-04
Epoch 4/300
35/35 [==============================] - 1s 21ms/step - loss: 0.6796 - mae: 0.6361 - val_loss: 0.4618 - val_mae: 0.5086 - lr: 5.0000e-04
Epoch 5/300
35/35 [==============================] - 1s 26ms/step - loss: 0.6199 - mae: 0.6059 - val_loss: 0.4547 - val_mae: 0.4962 - lr: 5.0000e-04
Epoch 6/300
35/3